In [52]:
import pandas as pd
import numpy as np
import re

In [53]:
file_path = "/content/smart meter  Tarrifs_validated_Final.xlsx"
sheet_name = "Validated Data"

df = pd.read_excel(file_path, sheet_name=sheet_name)
print(df.shape)
print(df.columns.tolist())
df.head(2)

(54, 58)
['Column1.id', 'provider_name', 'website_url', 'tariff_url', 'Meter_Type', 'tariff_name', 'tariffs.is_promotional', 'tariffs.promo_duration_months', 'standing_charges.urban', 'standing_charges.rural', 'Standing_charges_nightstorage_meter', 'standing_charge_period', 'standing_charge_includes_vat', 'tariffs.vat_included', 'vat_rate', 'rates_json.type', 'rate_time_00', 'rate_time_01', 'rate_time_02', 'rate_time_03', 'rate_time_04', 'rate_time_05', 'rate_time_06', 'rate_time_07', 'rate_time_08', 'rate_time_09', 'rate_time_10', 'rate_time_11', 'rate_time_12', 'rate_time_13', 'rate_time_14', 'rate_time_15', 'rate_time_16', 'rate_time_17', 'rate_time_18', 'rate_time_19', 'rate_time_20', 'rate_time_21', 'rate_time_22', 'rate_time_23', 'weekendfree_start_time', 'weekendfree_end_time', 'Nightstorage_meter_start', 'Nightstorage_meter_end', 'Nightstorage_rate', 'Welcome Bonus', 'rates_timeband_notes', 'rates_json.currency', 'rates_json.unit', 'has_feed_in_tariff', 'feed_in_rates_json.type

,Column1.id,provider_name,website_url,tariff_url,Meter_Type,tariff_name,tariffs.is_promotional,tariffs.promo_duration_months,standing_charges.urban,standing_charges.rural,...,rates_json.unit,has_feed_in_tariff,feed_in_rates_json.type,feed_in_rates_json.rate_c_kwh,reversion_tariff_name,source.verified,tariffs.source.retrieved_at,tariffs.human_verification,created_at,last_checked_at
0,1,Electric Ireland,https://www.electricireland.ie,https://www.electricireland.ie/switch/new-cust...,Smart Meter (SST),Home Electric + Saver 14%,True,12.0,250.77,314.98,...,c/kWh,False,NaN,NaN,Standard Tariff,True,2026-03-11,True,2026-03-11,2026-04-08
1,1,Electric Ireland,https://www.electricireland.ie,https://www.electricireland.ie/switch/new-cust...,Smart Meter (SST),Home Electric + Saver 14%,True,12.0,250.77,314.98,...,c/kWh,False,NaN,NaN,Standard Tariff,True,2026-03-11,True,2026-03-11,2026-04-08


In [54]:
plan_df = df.copy()

plan_df = plan_df.rename(columns={
    "provider_name": "supplier",
    "tariff_name": "plan_name",
    "Meter_Type": "meter_type",
    "tariffs.is_promotional": "is_promotional",
    "tariffs.promo_duration_months": "promo_duration_months",
    "standing_charges.urban": "standing_charge_urban",
    "standing_charges.rural": "standing_charge_rural",
    "Standing_charges_nightstorage_meter": "standing_charge_nightstorage",
    "standing_charge_period": "standing_charge_period",
    "standing_charge_includes_vat": "standing_charge_includes_vat",
    "tariffs.vat_included": "vat_included",
    "vat_rate": "vat_rate",
    "rates_json.type": "rate_type",
    "weekendfree_start_time": "weekendfree_start_time",
    "weekendfree_end_time": "weekendfree_end_time",
    "Nightstorage_meter_start": "nightstorage_start",
    "Nightstorage_meter_end": "nightstorage_end",
    "Nightstorage_rate": "nightstorage_rate",
    "Welcome Bonus": "welcome_bonus",
    "rates_timeband_notes": "rate_notes",
    "rates_json.currency": "currency",
    "rates_json.unit": "rate_unit",
    "has_feed_in_tariff": "has_feed_in_tariff",
    "feed_in_rates_json.type": "feed_in_type",
    "feed_in_rates_json.rate_c_kwh": "feed_in_rate_c_kwh",
    "reversion_tariff_name": "reversion_tariff_name",
    "source.verified": "source_verified",
    "tariffs.source.retrieved_at": "source_retrieved_at",
    "tariffs.human_verification": "human_verified",
})

# Add tariff variant label
def classify_variant(row):
    notes = str(row.get("rate_notes", "")).lower()
    if "post promo" in notes or "post-promo" in notes:
        return "post_promo"
    if bool(row.get("is_promotional")):
        return "promo"
    return "standard"

plan_df["tariff_variant"] = plan_df.apply(classify_variant, axis=1)

# Keep only needed summary columns
summary_cols = [
    "supplier",
    "plan_name",
    "meter_type",
    "tariff_variant",
    "is_promotional",
    "promo_duration_months",
    "rate_type",
    "standing_charge_urban",
    "standing_charge_rural",
    "standing_charge_nightstorage",
    "standing_charge_period",
    "standing_charge_includes_vat",
    "vat_included",
    "vat_rate",
    "weekendfree_start_time",
    "weekendfree_end_time",
    "nightstorage_start",
    "nightstorage_end",
    "nightstorage_rate",
    "welcome_bonus",
    "currency",
    "rate_unit",
    "has_feed_in_tariff",
    "feed_in_type",
    "feed_in_rate_c_kwh",
    "reversion_tariff_name",
    "source_verified",
    "human_verified",
    "rate_notes",
    "tariff_url",
    "website_url",
    "source_retrieved_at",
    "created_at",
    "last_checked_at"
]

plan_summary = plan_df[summary_cols].copy()
print(plan_summary.shape)
plan_summary.head(2)

(54, 34)


,supplier,plan_name,meter_type,tariff_variant,is_promotional,promo_duration_months,rate_type,standing_charge_urban,standing_charge_rural,standing_charge_nightstorage,...,feed_in_rate_c_kwh,reversion_tariff_name,source_verified,human_verified,rate_notes,tariff_url,website_url,source_retrieved_at,created_at,last_checked_at
0,Electric Ireland,Home Electric + Saver 14%,Smart Meter (SST),promo,True,12.0,standard,250.77,314.98,NaN,...,NaN,Standard Tariff,True,True,24-hour flat tariff — 00:00–24:00 (29.27c/kWh)...,https://www.electricireland.ie/switch/new-cust...,https://www.electricireland.ie,2026-03-11,2026-03-11,2026-04-08
1,Electric Ireland,Home Electric + Saver 14%,Smart Meter (SST),post_promo,True,12.0,standard,250.77,314.98,NaN,...,NaN,Standard Tariff,True,True,24-hour flat tariff — 00:00–24:00 (34.03c/kWh)...,https://www.electricireland.ie/switch/new-cust...,https://www.electricireland.ie,2026-03-11,2026-03-11,2026-04-08


In [55]:
hour_cols = [f"rate_time_{str(i).zfill(2)}" for i in range(24)]

id_cols = [
    "supplier",
    "plan_name",
    "meter_type",
    "tariff_variant",
    "is_promotional",
    "promo_duration_months",
    "rate_type",
    "standing_charge_urban",
    "standing_charge_rural",
    "standing_charge_nightstorage",
    "standing_charge_period",
    "standing_charge_includes_vat",
    "vat_included",
    "vat_rate",
    "weekendfree_start_time",
    "weekendfree_end_time",
    "nightstorage_start",
    "nightstorage_end",
    "nightstorage_rate",
    "welcome_bonus",
    "currency",
    "rate_unit",
    "has_feed_in_tariff",
    "feed_in_type",
    "feed_in_rate_c_kwh",
    "reversion_tariff_name",
    "source_verified",
    "human_verified",
    "rate_notes",
    "tariff_url",
    "website_url",
    "source_retrieved_at",
    "created_at",
    "last_checked_at"
]

hourly_df = plan_df[id_cols + hour_cols].melt(
    id_vars=id_cols,
    value_vars=hour_cols,
    var_name="hour_col",
    value_name="unit_rate_c_kwh"
)

hourly_df["hour"] = hourly_df["hour_col"].str.extract(r"(\d{2})").astype(int)
hourly_df["unit_rate_c_kwh"] = pd.to_numeric(hourly_df["unit_rate_c_kwh"], errors="coerce")

# Drop missing hourly rates
hourly_df = hourly_df.dropna(subset=["unit_rate_c_kwh"]).copy()

# Optional helper columns
hourly_df["unit_rate_eur_kwh"] = hourly_df["unit_rate_c_kwh"] / 100.0
hourly_df["plan_key"] = (
    hourly_df["supplier"].astype(str).str.strip() + " | " +
    hourly_df["plan_name"].astype(str).str.strip() + " | " +
    hourly_df["meter_type"].astype(str).str.strip() + " | " +
    hourly_df["tariff_variant"].astype(str).str.strip()
)

# Reorder columns
hourly_cols_final = [
    "plan_key",
    "supplier",
    "plan_name",
    "meter_type",
    "tariff_variant",
    "hour",
    "unit_rate_c_kwh",
    "unit_rate_eur_kwh",
    "rate_type",
    "standing_charge_urban",
    "standing_charge_rural",
    "standing_charge_nightstorage",
    "standing_charge_period",
    "standing_charge_includes_vat",
    "vat_included",
    "vat_rate",
    "weekendfree_start_time",
    "weekendfree_end_time",
    "nightstorage_start",
    "nightstorage_end",
    "nightstorage_rate",
    "welcome_bonus",
    "currency",
    "rate_unit",
    "has_feed_in_tariff",
    "feed_in_type",
    "feed_in_rate_c_kwh",
    "reversion_tariff_name",
    "source_verified",
    "human_verified",
    "rate_notes",
    "tariff_url",
    "website_url",
    "source_retrieved_at",
    "created_at",
    "last_checked_at"
]

hourly_df = hourly_df[hourly_cols_final].sort_values(
    ["supplier", "plan_name", "meter_type", "tariff_variant", "hour"]
).reset_index(drop=True)

print(hourly_df.shape)
hourly_df.head(10)

(1296, 36)


,plan_key,supplier,plan_name,meter_type,tariff_variant,hour,unit_rate_c_kwh,unit_rate_eur_kwh,rate_type,standing_charge_urban,...,feed_in_rate_c_kwh,reversion_tariff_name,source_verified,human_verified,rate_notes,tariff_url,website_url,source_retrieved_at,created_at,last_checked_at
0,Bord Gáis Energy | Electricity Discount 32% | ...,Bord Gáis Energy,Electricity Discount 32%,24 Hour Standard Meter,post_promo,0,41.59,0.4159,standard,244.76,...,18.5,Standard Tariff,True,True,Flat rate 41.59c/kWh; Post Promotional rates.,https://www.bordgaisenergy.ie/home/our-plans/a...,https://www.bordgaisenergy.ie,2026-03-11,2026-03-11,2026-04-08
1,Bord Gáis Energy | Electricity Discount 32% | ...,Bord Gáis Energy,Electricity Discount 32%,24 Hour Standard Meter,post_promo,1,41.59,0.4159,standard,244.76,...,18.5,Standard Tariff,True,True,Flat rate 41.59c/kWh; Post Promotional rates.,https://www.bordgaisenergy.ie/home/our-plans/a...,https://www.bordgaisenergy.ie,2026-03-11,2026-03-11,2026-04-08
2,Bord Gáis Energy | Electricity Discount 32% | ...,Bord Gáis Energy,Electricity Discount 32%,24 Hour Standard Meter,post_promo,2,41.59,0.4159,standard,244.76,...,18.5,Standard Tariff,True,True,Flat rate 41.59c/kWh; Post Promotional rates.,https://www.bordgaisenergy.ie/home/our-plans/a...,https://www.bordgaisenergy.ie,2026-03-11,2026-03-11,2026-04-08
3,Bord Gáis Energy | Electricity Discount 32% | ...,Bord Gáis Energy,Electricity Discount 32%,24 Hour Standard Meter,post_promo,3,41.59,0.4159,standard,244.76,...,18.5,Standard Tariff,True,True,Flat rate 41.59c/kWh; Post Promotional rates.,https://www.bordgaisenergy.ie/home/our-plans/a...,https://www.bordgaisenergy.ie,2026-03-11,2026-03-11,2026-04-08
4,Bord Gáis Energy | Electricity Discount 32% | ...,Bord Gáis Energy,Electricity Discount 32%,24 Hour Standard Meter,post_promo,4,41.59,0.4159,standard,244.76,...,18.5,Standard Tariff,True,True,Flat rate 41.59c/kWh; Post Promotional rates.,https://www.bordgaisenergy.ie/home/our-plans/a...,https://www.bordgaisenergy.ie,2026-03-11,2026-03-11,2026-04-08
5,Bord Gáis Energy | Electricity Discount 32% | ...,Bord Gáis Energy,Electricity Discount 32%,24 Hour Standard Meter,post_promo,5,41.59,0.4159,standard,244.76,...,18.5,Standard Tariff,True,True,Flat rate 41.59c/kWh; Post Promotional rates.,https://www.bordgaisenergy.ie/home/our-plans/a...,https://www.bordgaisenergy.ie,2026-03-11,2026-03-11,2026-04-08
6,Bord Gáis Energy | Electricity Discount 32% | ...,Bord Gáis Energy,Electricity Discount 32%,24 Hour Standard Meter,post_promo,6,41.59,0.4159,standard,244.76,...,18.5,Standard Tariff,True,True,Flat rate 41.59c/kWh; Post Promotional rates.,https://www.bordgaisenergy.ie/home/our-plans/a...,https://www.bordgaisenergy.ie,2026-03-11,2026-03-11,2026-04-08
7,Bord Gáis Energy | Electricity Discount 32% | ...,Bord Gáis Energy,Electricity Discount 32%,24 Hour Standard Meter,post_promo,7,41.59,0.4159,standard,244.76,...,18.5,Standard Tariff,True,True,Flat rate 41.59c/kWh; Post Promotional rates.,https://www.bordgaisenergy.ie/home/our-plans/a...,https://www.bordgaisenergy.ie,2026-03-11,2026-03-11,2026-04-08
8,Bord Gáis Energy | Electricity Discount 32% | ...,Bord Gáis Energy,Electricity Discount 32%,24 Hour Standard Meter,post_promo,8,41.59,0.4159,standard,244.76,...,18.5,Standard Tariff,True,True,Flat rate 41.59c/kWh; Post Promotional rates.,https://www.bordgaisenergy.ie/home/our-plans/a...,https://www.bordgaisenergy.ie,2026-03-11,2026-03-11,2026-04-08
9,Bord Gáis Energy | Electricity Discount 32% | ...,Bord Gáis Energy,Electricity Discount 32%,24 Hour Standard Meter,post_promo,9,41.59,0.4159,standard,244.76,...,18.5,Standard Tariff,True,True,Flat rate 41.59c/kWh; Post Promotional rates.,https://www.bordgaisenergy.ie/home/our-plans/a...,https://www.bordgaisenergy.ie,2026-03-11,2026-03-11,2026-04-08


In [56]:
plan_summary.to_csv("cleaned_tariff_plan_summary.csv", index=False)
hourly_df.to_csv("cleaned_tariff_hourly.csv", index=False)

print("Saved cleaned_tariff_plan_summary.csv")
print("Saved cleaned_tariff_hourly.csv")

Saved cleaned_tariff_plan_summary.csv
Saved cleaned_tariff_hourly.csv


In [57]:
print("Unique suppliers:", hourly_df["supplier"].nunique())
print("Unique plans:", hourly_df[["supplier", "plan_name", "meter_type", "tariff_variant"]].drop_duplicates().shape[0])

check_counts = (
    hourly_df.groupby(["supplier", "plan_name", "meter_type", "tariff_variant"])["hour"]
    .nunique()
    .reset_index(name="hour_count")
)

print(check_counts["hour_count"].value_counts().sort_index())
check_counts.head(10)

Unique suppliers: 5
Unique plans: 53
hour_count
24    53
Name: count, dtype: int64


,supplier,plan_name,meter_type,tariff_variant,hour_count
0,Bord Gáis Energy,Electricity Discount 32%,24 Hour Standard Meter,post_promo,24
1,Bord Gáis Energy,Electricity Discount 32%,24 Hour Standard Meter,promo,24
2,Bord Gáis Energy,Smart All Day Electricity Discount,Smart Meter,post_promo,24
3,Bord Gáis Energy,Smart All Day Electricity Discount,Smart Meter,promo,24
4,Bord Gáis Energy,Smart EV Electricity Discount,Smart Meter,post_promo,24
5,Bord Gáis Energy,Smart EV Electricity Discount,Smart Meter,promo,24
6,Bord Gáis Energy,Smart Free Saturday Electricity Discount,Smart Meter,post_promo,24
7,Bord Gáis Energy,Smart Free Saturday Electricity Discount,Smart Meter,promo,24
8,Bord Gáis Energy,Smart Free Sunday Electricity Discount,Smart Meter,post_promo,24
9,Bord Gáis Energy,Smart Free Sunday Electricity Discount,Smart Meter,promo,24


In [58]:
from google.colab import files

files.download("cleaned_tariff_hourly.csv")
files.download("cleaned_tariff_plan_summary.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Part b


In [60]:
import os

os.makedirs("data", exist_ok=True)
print("data folder created")

data folder created


In [61]:
sample_usage = pd.DataFrame({
    "datetime": [
        "2026-04-01 00:00:00",
        "2026-04-01 00:30:00",
        "2026-04-01 01:00:00",
        "2026-04-01 01:30:00",
        "2026-04-01 02:00:00",
        "2026-04-01 02:30:00"
    ],
    "demand": [1.25, 1.10, 0.95, 0.90, 0.88, 0.85],
    "temp": [10.2, 10.0, 9.8, 9.6, 9.4, 9.2]
})

sample_usage.to_csv("data/sample_user_usage.csv", index=False)

print("sample_user_usage.csv created successfully")

sample_user_usage.csv created successfully


In [62]:
import shutil

shutil.move("cleaned_tariff_hourly.csv", "data/cleaned_tariff_hourly.csv")
shutil.move("cleaned_tariff_plan_summary.csv", "data/cleaned_tariff_plan_summary.csv")

print("Tariff files moved into data folder")

Tariff files moved into data folder


In [63]:
import pandas as pd

usage_df = pd.read_csv("data/sample_user_usage.csv")
tariff_df = pd.read_csv("data/cleaned_tariff_hourly.csv")

usage_df["datetime"] = pd.to_datetime(usage_df["datetime"], errors="coerce")
usage_df["demand"] = pd.to_numeric(usage_df["demand"], errors="coerce")

usage_df = usage_df.dropna(subset=["datetime", "demand"]).copy()
usage_df["hour"] = usage_df["datetime"].dt.hour

tariff_df["hour"] = pd.to_numeric(tariff_df["hour"], errors="coerce")
tariff_df["unit_rate_eur_kwh"] = pd.to_numeric(tariff_df["unit_rate_eur_kwh"], errors="coerce")

In [64]:
def compare_supplier_costs_30min(usage_df, tariff_df, standing_charge_type="urban"):
    results = []

    plan_keys = tariff_df["plan_key"].dropna().unique()

    for plan_key in plan_keys:
        plan_rates = tariff_df[tariff_df["plan_key"] == plan_key].copy()

        merged = usage_df.merge(
            plan_rates[[
                "plan_key",
                "supplier",
                "plan_name",
                "meter_type",
                "tariff_variant",
                "hour",
                "unit_rate_eur_kwh",
                "standing_charge_urban",
                "standing_charge_rural",
                "welcome_bonus"
            ]],
            on="hour",
            how="left"
        )

        if merged["unit_rate_eur_kwh"].isna().all():
            continue

        merged["interval_cost"] = merged["demand"] * merged["unit_rate_eur_kwh"]

        energy_cost = merged["interval_cost"].sum()

        if standing_charge_type == "rural":
            standing_charge_annual = pd.to_numeric(merged["standing_charge_rural"], errors="coerce").dropna().iloc[0]
        else:
            standing_charge_annual = pd.to_numeric(merged["standing_charge_urban"], errors="coerce").dropna().iloc[0]

        standing_charge_daily = standing_charge_annual / 365 if pd.notna(standing_charge_annual) else 0

        days_covered = max(1, usage_df["datetime"].dt.date.nunique())
        standing_cost = standing_charge_daily * days_covered

        total_cost = energy_cost + standing_cost

        first_row = merged.iloc[0]

        results.append({
            "plan_key": plan_key,
            "supplier": first_row["supplier"],
            "plan_name": first_row["plan_name"],
            "meter_type": first_row["meter_type"],
            "tariff_variant": first_row["tariff_variant"],
            "days_covered": days_covered,
            "energy_cost_eur": round(energy_cost, 2),
            "standing_cost_eur": round(standing_cost, 2),
            "total_cost_eur": round(total_cost, 2),
            "avg_daily_cost_eur": round(total_cost / days_covered, 2),
        })

    return pd.DataFrame(results).sort_values("total_cost_eur").reset_index(drop=True)

In [65]:
comparison_df = compare_supplier_costs_30min(
    usage_df=usage_df,
    tariff_df=tariff_df,
    standing_charge_type="urban"
)

comparison_df.head(10)

,plan_key,supplier,plan_name,meter_type,tariff_variant,days_covered,energy_cost_eur,standing_cost_eur,total_cost_eur,avg_daily_cost_eur
0,SSE Airtricity | Smart EV Max Electricity 30% ...,SSE Airtricity,Smart EV Max Electricity 30% Discount,Smart EV Meter,promo,1,0.72,0.98,1.70,1.70
1,Electric Ireland | Home Electric+ Night Boost ...,Electric Ireland,Home Electric+ Night Boost,Smart Meter (SST),promo,1,0.88,0.90,1.78,1.78
2,Electric Ireland | Home Electric+ Night Boost ...,Electric Ireland,Home Electric+ Night Boost,Smart Meter (SST),post_promo,1,0.93,0.90,1.83,1.83
3,Energia | EV Smart Drive Plus | Smart Meter | ...,Energia,EV Smart Drive Plus,Smart Meter,promo,1,1.20,0.73,1.92,1.92
4,Electric Ireland | Electricity NightSaver 5.5%...,Electric Ireland,Electricity NightSaver 5.5%,Day and Night Meter,promo,1,1.03,0.90,1.93,1.93
5,Electric Ireland | Green Electricity NightSave...,Electric Ireland,Green Electricity NightSaver 5.5%,Day and Night Meter,promo,1,1.04,0.90,1.94,1.94
6,Energia | Energia SST | Smart Meter | promo,Energia,Energia SST,Smart Meter,promo,1,1.25,0.73,1.97,1.97
7,SSE Airtricity | Smart Day / Night / Peak - El...,SSE Airtricity,Smart Day / Night / Peak - Electricity - Top d...,Smart Meter,promo,1,1.24,0.72,1.97,1.97
8,Bord Gáis Energy | Smart Standard Electricity ...,Bord Gáis Energy,Smart Standard Electricity Discount,Smart Meter,promo,1,1.32,0.67,1.99,1.99
9,Electric Ireland | Electricity NightSaver 5.5%...,Electric Ireland,Electricity NightSaver 5.5%,Day and Night Meter,post_promo,1,1.09,0.90,1.99,1.99


In [66]:
best_plan = comparison_df.iloc[0]
print("Cheapest supplier:", best_plan["supplier"])
print("Cheapest plan:", best_plan["plan_name"])
print("Estimated total cost (€):", best_plan["total_cost_eur"])

Cheapest supplier: SSE Airtricity
Cheapest plan: Smart EV Max Electricity 30% Discount
Estimated total cost (€): 1.7


In [67]:
def simulate_shift_savings_30min(usage_df, tariff_df, target_shift_hour=2, shift_fraction=0.2, standing_charge_type="urban"):
    results = []

    usage_shifted = usage_df.copy()
    usage_shifted["hour"] = usage_shifted["datetime"].dt.hour

    # Define expensive peak window
    peak_mask = usage_shifted["hour"].isin([17, 18, 19])

    shift_amount = usage_shifted.loc[peak_mask, "demand"] * shift_fraction
    usage_shifted.loc[peak_mask, "demand"] = usage_shifted.loc[peak_mask, "demand"] - shift_amount

    # Add shifted usage to target hour rows
    target_mask = usage_shifted["hour"] == target_shift_hour
    if target_mask.sum() > 0:
        add_per_row = shift_amount.sum() / target_mask.sum()
        usage_shifted.loc[target_mask, "demand"] = usage_shifted.loc[target_mask, "demand"] + add_per_row

    base_costs = compare_supplier_costs_30min(usage_df, tariff_df, standing_charge_type)
    shifted_costs = compare_supplier_costs_30min(usage_shifted, tariff_df, standing_charge_type)

    merged = base_costs.merge(
        shifted_costs[["plan_key", "total_cost_eur"]],
        on="plan_key",
        suffixes=("_base", "_shifted")
    )

    merged["savings_eur"] = merged["total_cost_eur_base"] - merged["total_cost_eur_shifted"]
    merged = merged.sort_values("savings_eur", ascending=False).reset_index(drop=True)

    return merged, usage_shifted

In [68]:
shift_results_df, shifted_usage_df = simulate_shift_savings_30min(
    usage_df=usage_df,
    tariff_df=tariff_df,
    target_shift_hour=2,
    shift_fraction=0.2,
    standing_charge_type="urban"
)

shift_results_df.head(10)

,plan_key,supplier,plan_name,meter_type,tariff_variant,days_covered,energy_cost_eur,standing_cost_eur,total_cost_eur_base,avg_daily_cost_eur,total_cost_eur_shifted,savings_eur
0,SSE Airtricity | Smart EV Max Electricity 30% ...,SSE Airtricity,Smart EV Max Electricity 30% Discount,Smart EV Meter,promo,1,0.72,0.98,1.70,1.70,1.70,0.0
1,Electric Ireland | Home Electric+ Night Boost ...,Electric Ireland,Home Electric+ Night Boost,Smart Meter (SST),promo,1,0.88,0.90,1.78,1.78,1.78,0.0
2,Electric Ireland | Home Electric+ Night Boost ...,Electric Ireland,Home Electric+ Night Boost,Smart Meter (SST),post_promo,1,0.93,0.90,1.83,1.83,1.83,0.0
3,Energia | EV Smart Drive Plus | Smart Meter | ...,Energia,EV Smart Drive Plus,Smart Meter,promo,1,1.20,0.73,1.92,1.92,1.92,0.0
4,Electric Ireland | Electricity NightSaver 5.5%...,Electric Ireland,Electricity NightSaver 5.5%,Day and Night Meter,promo,1,1.03,0.90,1.93,1.93,1.93,0.0
5,Electric Ireland | Green Electricity NightSave...,Electric Ireland,Green Electricity NightSaver 5.5%,Day and Night Meter,promo,1,1.04,0.90,1.94,1.94,1.94,0.0
6,Energia | Energia SST | Smart Meter | promo,Energia,Energia SST,Smart Meter,promo,1,1.25,0.73,1.97,1.97,1.97,0.0
7,SSE Airtricity | Smart Day / Night / Peak - El...,SSE Airtricity,Smart Day / Night / Peak - Electricity - Top d...,Smart Meter,promo,1,1.24,0.72,1.97,1.97,1.97,0.0
8,Bord Gáis Energy | Smart Standard Electricity ...,Bord Gáis Energy,Smart Standard Electricity Discount,Smart Meter,promo,1,1.32,0.67,1.99,1.99,1.99,0.0
9,Electric Ireland | Electricity NightSaver 5.5%...,Electric Ireland,Electricity NightSaver 5.5%,Day and Night Meter,post_promo,1,1.09,0.90,1.99,1.99,1.99,0.0
